# Data Exploration

This notebook verifies the DeepGlobe Road Extraction dataset structure, visualizes satellite image/mask pairs, and creates an 80/10/10 train-validation-test split from the labeled training data.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split

DATA_DIR = Path("../data")
TRAIN_DIR = DATA_DIR / "train"

In [ ]:
image_paths = sorted(TRAIN_DIR.glob("*_sat.jpg"))
mask_paths = sorted(TRAIN_DIR.glob("*_mask.png"))

pairs_match = all(
    img.name.replace("_sat.jpg", "") == mask.name.replace("_mask.png", "")
    for img, mask in zip(image_paths, mask_paths)
)

len(image_paths), len(mask_paths), pairs_match, image_paths[:3], mask_paths[:3]

In [ ]:
idx = 0

img = Image.open(image_paths[idx]).convert("RGB")
mask = Image.open(mask_paths[idx]).convert("L")

img_np = np.array(img)
mask_np = np.array(mask)

plt.figure(figsize=(16, 5))

plt.subplot(1, 3, 1)
plt.imshow(img_np)
plt.title("Satellite Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mask_np, cmap="gray")
plt.title("Road Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(img_np)
plt.imshow(mask_np, cmap="Reds", alpha=0.4)
plt.title("Overlay")
plt.axis("off")

plt.show()

img.size, mask.size

In [ ]:
train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(
    image_paths,
    mask_paths,
    test_size=0.2,
    random_state=42
)

val_imgs, test_imgs, val_masks, test_masks = train_test_split(
    temp_imgs,
    temp_masks,
    test_size=0.5,
    random_state=42
)

len(train_imgs), len(val_imgs), len(test_imgs)

In [ ]:
rows = []

for split_name, imgs, masks in [
    ("train", train_imgs, train_masks),
    ("val", val_imgs, val_masks),
    ("test", test_imgs, test_masks),
]:
    for img_path, mask_path in zip(imgs, masks):
        rows.append({
            "split": split_name,
            "image_path": str(img_path),
            "mask_path": str(mask_path)
        })

df = pd.DataFrame(rows)
df.to_csv("../data/splits.csv", index=False)

df["split"].value_counts()